# DS2002 · Midterm Progress

**Lab — 2026-10-30 · Fall 2026**  
**Team:** Viet Huynh (solo, with instructor approval)

This checkpoint uses the supplied September 2024 Walmart files. The 7-day pre-landfall window is Sept. 19–25, 2024; the preceding 7-day baseline is Sept. 12–18. Hurricane Helene made Florida landfall on Sept. 26, 2024.

---

## Lab 10 — Midterm progress, week 2 of 3

One submission per team. Questions 1 through 3 should be substantially answered by now — not polished, but answered, with a number and a chart each.

Presentations are Monday. If a question is not working, this is the last checkpoint where there is time to do something about it, so report the actual state rather than the intended one.

### Q1 — Demand surge

**TODO:** your baseline window, your surge window, the average for each, and the multiple. Plus the chart. State the window boundaries explicitly — a multiple without a window is not a finding.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path(r"C:\Users\Viet\Downloads")
tx_raw = pd.read_csv(DATA / "walmart_transactions.csv")
catalog = pd.read_csv(DATA / "product_catalog.csv")

# Clean exact duplicates and malformed/missing timestamps. Retain the original
# negative quantity flag, then use absolute units for this sales-volume analysis.
n_start = len(tx_raw)
tx = tx_raw.drop_duplicates().copy()
n_dedup = len(tx)
tx["timestamp"] = pd.to_datetime(tx["timestamp"], errors="coerce", format="mixed")
n_bad_time = int(tx["timestamp"].isna().sum())
tx = tx.dropna(subset=["timestamp"]).copy()
tx["date"] = tx["timestamp"].dt.normalize()
tx["sku_norm"] = tx["sku"].astype(str).str.strip().str.upper()
tx["quantity"] = pd.to_numeric(tx["quantity"], errors="coerce").abs()
tx = tx.dropna(subset=["quantity"])

sku_category = dict(zip(catalog["sku"].str.strip().str.upper(),
                        catalog["category"].str.strip().str.casefold()))
tx["category"] = tx["sku_norm"].map(sku_category).fillna(tx["category"].astype(str).str.strip().str.casefold())
aliases = {"breakfast foods":"breakfast", "baby products":"baby",
           "games & entertainment":"entertainment", "household essentials":"household",
           "groceries":"grocery", "emergency supplies":"emergency",
           "alcoholic beverages":"alcohol", "health & wellness":"health"}
tx["category"] = tx["category"].replace(aliases)

daily = tx.groupby(["date","category"])["quantity"].sum().unstack(fill_value=0)
baseline = daily.loc["2024-09-12":"2024-09-18"].mean()
surge = daily.loc["2024-09-19":"2024-09-25"].mean()
q1 = pd.DataFrame({"baseline_units_per_day":baseline, "surge_units_per_day":surge})
q1["increase_pct"] = 100*(q1["surge_units_per_day"]/q1["baseline_units_per_day"]-1)
q1["multiple"] = q1["surge_units_per_day"]/q1["baseline_units_per_day"]
print(f"Rows: {n_start:,} raw; {n_start-n_dedup:,} exact duplicates removed; {n_bad_time:,} invalid timestamps dropped; {len(tx):,} analyzed.")
print("Baseline: Sept. 12–18, 2024. Pre-landfall: Sept. 19–25, 2024.")
display(q1.sort_values("increase_pct", ascending=False).round(2))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt

top5 = q1.nlargest(5, "increase_pct").index
ax = daily.loc["2024-09-12":"2024-09-25", top5].plot(figsize=(10,5), marker="o")
ax.axvspan(pd.Timestamp("2024-09-12"), pd.Timestamp("2024-09-18"), color="gray", alpha=.12, label="Baseline")
ax.axvspan(pd.Timestamp("2024-09-19"), pd.Timestamp("2024-09-25"), color="red", alpha=.10, label="7 days before landfall")
ax.set(title="Breakfast and emergency units rose most before Helene landfall", xlabel="Date", ylabel="Units sold")
ax.legend(title="Category"); ax.grid(alpha=.2); plt.tight_layout(); plt.show()

**Q1 answer, in one sentence a manager could read:** From Sept. 12–18 to Sept. 19–25, breakfast averaged 110.1 versus 378.0 units per day, a 243% increase (3.43×), the largest category increase; emergency supplies rose 137% (426.4 to 1,010.6 units/day).

### Q2 — SKU consolidation

**TODO:** how many distinct SKUs you started with, how many canonical products you ended with, your largest consolidated family, and a before-and-after comparison showing what changes. Include the assertion proving consolidation did not change unit totals.

In [ ]:
import matplotlib.pyplot as plt

# Reviewed SKU-to-family mapping based on the supplied catalog's product names.
families = {
 "Pop-Tarts Strawberry":["PT-12","SKU#459812","POPTART-STRAW","PT_STRAWBERRY"],
 "Bottled water":["WTR-24","WATER24PK","WTR_24PK"],
 "AA batteries":["BATT-100","BAT-AA4","BATT_AA_4PK"],
 "LED flashlight":["FLASH-01","LED-FLASH"],
 "Light beer":["BEER-24","LTBEER24"],
 "Monopoly board game":["BG-MONO","MONOPOLY-01","GAME-MONO"],
 "Playing cards":["CARDS-01","PLAYCARD-STD"],
 "Streaming device":["STREAM-01","ROKU-STD"],
 "Canned soup":["SOUP-CAN","CNSOUP-01"],
 "White bread":["BREAD-WH","WHITEBREAD"],
 "Peanut butter":["PB-16","PNUTBTR-16"],
 "Emergency candles":["CNDL-EM","CANDLE-EMRG"],
 "First-aid kit":["FAID-01","FIRSTAID-KIT"],
 "Portable charger":["CHRG-PORT","PORTCHARGER"],
 "Chips variety pack":["CHIPS-VP","CHIPVARIETY"],
 "Paper towels":["PTOWEL-6","PAPERTOWEL6"],
 "Toilet paper":["TP-12","TOILETPAPER12"],
 "Diapers size 3":["DIAP-S3","DIAPERS-3"],
 "Infant formula":["BFORM-01","BABYFORMULA"],
 "Heavy-duty tarp":["TARP-HD","HDTARP-01"],
}
sku_family={sku.upper():family for family,skus in families.items() for sku in skus}
raw_skus=tx.copy()
raw_skus["canonical_id"]=raw_skus["sku_norm"].map(sku_family)
assert raw_skus["canonical_id"].notna().all()
consolidated=raw_skus.groupby("canonical_id",as_index=False)["quantity"].sum()
initial_skus=raw_skus["sku_norm"].nunique()
final_products=consolidated["canonical_id"].nunique()
assert raw_skus["quantity"].sum()==consolidated["quantity"].sum()
pt=raw_skus[raw_skus["canonical_id"]=="Pop-Tarts Strawberry"]
pt_by_sku=pt.groupby("sku_norm")["quantity"].sum().sort_values()
pt_daily=pt.groupby("date")["quantity"].sum()
print(f"Distinct transaction SKUs: {initial_skus}; canonical families: {final_products}")
print(f"Largest family: Pop-Tarts Strawberry ({len(families['Pop-Tarts Strawberry'])} SKUs)")
print(f"Unit totals preserved: {raw_skus.quantity.sum():,.0f} before = {consolidated.quantity.sum():,.0f} after")
print(f"Pop-Tarts: {pt.quantity.sum():,.0f} units over {pt.date.nunique()} days; {pt_daily.mean():.1f} units/day on average")
fig, axes=plt.subplots(1,2,figsize=(12,4))
pt_by_sku.plot.barh(ax=axes[0],title="Fragmented Pop-Tarts units by SKU")
pt_daily.plot(ax=axes[1],marker="o",title="Consolidated Pop-Tarts daily units")
axes[0].set_xlabel("Units"); axes[1].set_ylabel("Units"); axes[1].set_xlabel("Date")
plt.tight_layout(); plt.show()

**Q2 answer:** The transaction file contains 45 distinct SKUs, mapped into 20 canonical product families. Pop-Tarts Strawberry is the largest fragmented family, with four SKUs; consolidation preserves the total units (the assertion in code checks this) and yields one daily demand series, avoiding an understated replenishment signal when labels are split.

### Q3 — Weather correlation

**TODO:** the API call, the validated join, and the relationship you found. Include:

- the row count before and after the join, proving nothing was lost
- whether the data is live or fallback
- how many days the correlation is computed over

If your n is small, say so here rather than being asked about it on Monday.

In [ ]:
# Prepared live Open-Meteo Historical API call. This execution environment
# refused outbound API connections, so no weather result is claimed here.
import requests, time

weather_url = "https://archive-api.open-meteo.com/v1/archive"
params = {"latitude":27.9506, "longitude":-82.4572,
          "start_date":"2024-09-01", "end_date":"2024-09-30",
          "hourly":"temperature_2m,precipitation,wind_speed_10m,pressure_msl",
          "timezone":"America/New_York"}
payload = None
for attempt in range(3):
    try:
        response=requests.get(weather_url,params=params,timeout=30)
        response.raise_for_status()
        payload=response.json()
        break
    except requests.RequestException as exc:
        if attempt==2:
            print("API request not completed in this environment:", repr(exc))
        else:
            time.sleep(2**attempt)

if payload is not None:
    hourly=pd.DataFrame(payload["hourly"]).rename(columns={"time":"timestamp"})
    hourly["timestamp"]=pd.to_datetime(hourly["timestamp"])
    hourly["date"]=hourly["timestamp"].dt.normalize()
    weather=hourly.groupby("date",as_index=False).agg(
        temperature_2m_mean=("temperature_2m","mean"),
        precipitation_sum=("precipitation","sum"),
        wind_speed_10m_max=("wind_speed_10m","max"),
        pressure_msl_mean=("pressure_msl","mean"))
    sales_daily=tx.groupby("date",as_index=False)["quantity"].sum().rename(columns={"quantity":"units"})
    joined=sales_daily.merge(weather,on="date",how="left",validate="one_to_one")
    preserved=len(joined)==len(sales_daily) and joined["temperature_2m_mean"].notna().all()
    assert preserved, "Weather join lost or failed to match sales dates"
    corr=joined[["units","temperature_2m_mean","precipitation_sum","wind_speed_10m_max","pressure_msl_mean"]].corr()["units"]
    print(f"Sales rows before/after join: {len(sales_daily)} -> {len(joined)}; all dates matched: {preserved}")
    print(f"Live source: Open-Meteo Archive; days in correlation: {joined.date.nunique()}")
    display(corr.round(3))
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].scatter(joined["wind_speed_10m_max"],joined["units"])
    axes[0].set(title="Wind speed vs daily units",xlabel="Max wind speed (km/h)",ylabel="Units")
    axes[1].scatter(joined["precipitation_sum"],joined["units"])
    axes[1].set(title="Precipitation vs daily units",xlabel="Precipitation (mm)",ylabel="Units")
    plt.tight_layout(); plt.show()
else:
    print("Q3 correlation/chart not produced: live API access is required; no fallback data were substituted.")

**Q3 answer, with its caveat:** The notebook contains a real Open-Meteo Archive API request for Tampa-area hourly temperature, precipitation, wind, and pressure, with retries, local-time daily aggregation, and a validated join. The request was refused in this environment, so the correlation, matched-row count, and chart are explicitly not reported; no fallback or hand-built weather values were used. Run this cell with internet access to complete Q3.

### Q4 and Q5 — where they stand

**TODO:** one honest sentence each on the remaining two questions and who is finishing them this weekend.

**Q4 (stores and geography):** Not completed at this checkpoint. The supplied store file has 14 rows, including duplicate-form IDs and one location (Tallahassee) lacks coordinates; I have not yet compared store-level surge timing or geography. Viet owns the remaining work as the solo team member.

**Q5 (signal or artifact):** Not completed at this checkpoint. The SQLite database is available, but its board-game sales/inventory evidence has not yet been compared with transaction records. Viet owns the remaining work.

### Self-check

In [ ]:
progress = {
    'q1_multiple': round(float(q1.loc["breakfast","multiple"]),2),
    'q1_baseline_window': '2024-09-12 to 2024-09-18',
    'q2_canonical_products': int(final_products),
    'q3_join_rows_preserved': None,   # pending successful live API run
    'q3_days_in_correlation': None,   # pending API result
    'charts_with_message_titles': 2, # Q1 and Q2; Q3 chart awaits API
    'notebook_runs_clean': False,     # API cell still needs network-enabled run
    'repo_pushed': False,
}
for label,value in progress.items():
    print('PASS' if value is not None and value is not False else 'PENDING', '-', label, '=', value)
print()
print('ready for presentations:', False)

### Write-up

**a)** One finding that surprised your team, and what you checked to make sure it was real rather than a bug.

**b)** One number you reported that you are least confident in, and why.

a) Breakfast category sales grew the most in the seven days before landfall. I checked the increase across all seven daily totals and normalized category labels through the supplied catalog; the increase is spread across the window and not caused by one duplicate transaction.

b) I am least confident in any weather relationship because the live API request could not run in this environment. The planned comparison uses one Tampa-area weather series against chain-wide sales for 30 days, so even after it runs it will be descriptive and cannot establish that weather caused demand.